## 1. Clone the public GitHub repository (no ZIP upload)

After publishing, set your GitHub username and repository name in the next cell. This loads the Python scripts and data files.


In [ ]:
# Update these two values after publishing the repository.
GITHUB_USERNAME = "YOUR-USERNAME"
REPO_NAME = "PAIR-Reproducibility"

from pathlib import Path
import subprocess

if GITHUB_USERNAME == "YOUR-USERNAME":
    raise ValueError("Set GITHUB_USERNAME to your GitHub username first.")
repo_dir = Path("/content") / REPO_NAME
if not (repo_dir / ".git").exists():
    subprocess.run(["git", "clone", f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git", str(repo_dir)], check=True)
%cd {repo_dir}
%pip -q install -r requirements.txt
print("Project files loaded from GitHub.")


## 2. Connect Google Drive

**All experiment checkpoints go to `MyDrive/PAIR_Project_Results/`.** Do not delete or rename this folder between sessions. Avoid editing the checkpoint files manually.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_BASE = Path('/content/drive/MyDrive/PAIR_Project_Results')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
EXT_FOLDER = DRIVE_BASE / 'local_extension'
REP_FOLDER = DRIVE_BASE / 'local_reproduction'
print('Permanent results folder:', DRIVE_BASE)


## 3. Verify the checked-in resumable runner

`run_experiment_resumable.py` is part of this GitHub repository. Keep the implementation in a normal `.py` file so the code you submit is the code you execute.


In [ ]:
from pathlib import Path
runner = Path("run_experiment_resumable.py")
print("Resumable runner exists:", runner.exists())
assert runner.exists(), "Clone the repo using Step 1 first"


## 4. Check your Colab GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No NVIDIA GPU assigned.')

## 5. Download the official JBB dataset

Use the same dataset file in each session so saved checkpoints can be matched safely.

In [ ]:
from pathlib import Path
import urllib.request
url = 'https://huggingface.co/datasets/JailbreakBench/JBB-Behaviors/resolve/main/data/harmful-behaviors.csv'
Path('data').mkdir(exist_ok=True)
urllib.request.urlretrieve(url, 'data/harmful-behaviors.csv')
from run_experiment import load_behaviors, choose_20
behaviors = load_behaviors('data/harmful-behaviors.csv')
selection = choose_20(behaviors)
print('All behaviors:', len(behaviors))
print('Extension behaviors:', len(selection))
print('Categories:', len(set(x['category'] for x in selection)))

## 6. Optional: offline code tests (not empirical results)

In [ ]:
!python -m unittest discover -s tests
!python run_experiment.py --mode demo --task all --seeds 0 1 2 --out results/demo_run
!python analyze_results.py --input results/demo_run
print('These demo outputs are simulated, not real LLM results.')

## 7. Start Ollama and load the models

Use the same attacker/target and judge model tags as your previous runs. The runtime may need reinstallation after Colab disconnects.

In [ ]:
!apt-get update -qq
!apt-get install -y zstd

In [ ]:

import os
import time
import shutil
import subprocess
import requests

# Install Ollama if it is missing
if shutil.which("ollama") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
    subprocess.run(
        "curl -fsSL https://ollama.com/install.sh -o /tmp/ollama.sh && bash /tmp/ollama.sh",
        shell=True,
        check=True
    )

# Stop previous server if one exists
subprocess.run(["pkill", "-f", "^ollama serve"], check=False)
time.sleep(2)

# Configure Ollama
env = os.environ.copy()
env["OLLAMA_MAX_LOADED_MODELS"] = "2"
env["OLLAMA_NUM_PARALLEL"] = "1"
env["OLLAMA_KEEP_ALIVE"] = "5m"

# Start Ollama server
log = open("/tmp/ollama.log", "a")

server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log,
    stderr=subprocess.STDOUT,
    env=env
)

# Wait until the server is ready
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=3)
        if r.status_code == 200:
            print("Ollama is ready!")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start")


In [ ]:
!ollama pull llama3.2:3b
!ollama pull llama-guard3:1b
!ollama list

### Optional pilot: two behaviors

This is only a small real-model check. Run it if models were reinstalled or the server has been unstable.

In [ ]:
import subprocess
subprocess.run([
    'python', '-u', 'run_experiment_resumable.py',
    '--mode', 'ollama', '--allow-local-tests', '--task', 'extension',
    '--limit', '2', '--seeds', '0',
    '--data', 'data/harmful-behaviors.csv', '--out', 'results/local_pilot'
], check=True)
subprocess.run(['python','analyze_results.py','--input','results/local_pilot'],check=True)


In [ ]:

import os
import subprocess
import time
import requests

# Stop previous Ollama server
subprocess.run(
    ["pkill", "-f", "^ollama serve"],
    check=False
)
time.sleep(2)

# Ollama settings
env = os.environ.copy()
env["OLLAMA_MAX_LOADED_MODELS"] = "1"
env["OLLAMA_NUM_PARALLEL"] = "1"
env["OLLAMA_KEEP_ALIVE"] = "5m"

# Restart server
log = open("/tmp/ollama.log", "a")

server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True
)

# Wait for server
for i in range(30):
    try:
        r = requests.get(
            "http://localhost:11434/api/tags",
            timeout=3
        )
        if r.status_code == 200:
            print("Ollama is ready!")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start")


In [ ]:
!pwd
!ollama list
!curl -s http://localhost:11434/api/tags
!tail -n 20 /tmp/ollama.log

## 8. Extension: run ONE resumable batch each time

**Same planned experiment:** 20 selected behaviors × breadth 10×3 / depth 5×6 × seeds 0, 1, 2 = **120 behavior-level runs**.

- Click this cell **once** for the next unfinished batch (5 behaviors for both setups, one seed; up to 10 behavior-level runs).
- It automatically finds the first batch that is not yet complete; **you don't need to edit `start` or `seed`**.
- Every completed behavior is written to **Google Drive immediately**. Results CSVs are updated at the end of the batch or if a later behavior crashes.
- After a Colab disconnect, rerun **steps 1, 2, 3, 5, 7**, then click this same batch cell again. `SKIP ... (saved)` means the checkpoint was reused.
- If batches are still too long, change `batch_size = 5` to `2` **before the first batch**, or use `batch_size = 1` for subsequent reruns (identical behavior order).

In [ ]:
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/PAIR_Project_Results/local_extension/checkpoints"
)

saved = list(folder.glob("*.json"))
print(f"Recovered: {len(saved)}/120 runs")

In [ ]:
!nvidia-smi
!ollama list
!curl -sS --max-time 5 http://localhost:11434/api/tags

In [ ]:
import subprocess

cmd = [
    "python", "-u", "run_experiment_resumable.py",
    "--mode", "ollama",
    "--allow-local-tests",
    "--task", "extension",
    "--seeds", "1",
    "--start", "0",
    "--limit", "15",
    "--data", "data/harmful-behaviors.csv",
    "--out", "/content/drive/MyDrive/PAIR_Project_Results/local_extension"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("Exit code:", result.returncode)
print("OUTPUT:", result.stdout[-3000:])
print("ERROR:", result.stderr[-5000:])

In [ ]:
import subprocess
from pathlib import Path
from run_experiment_resumable import load_behaviors, choose_20, checkpoint_path

batch_size = 5            # Set to 2 or 1 for shorter sessions
seeds = [0, 1, 2]
selected = choose_20(load_behaviors('data/harmful-behaviors.csv'))
EXT_FOLDER.mkdir(parents=True, exist_ok=True)

next_batch = None
for seed in seeds:
    for start in range(0, len(selected), batch_size):
        chunk = selected[start:start + batch_size]
        missing = [
            (setup, b['id'])
            for setup in ('breadth', 'depth') for b in chunk
            if not checkpoint_path(EXT_FOLDER, setup, seed, b['id']).exists()
        ]
        if missing:
            next_batch = (seed, start, len(chunk), len(missing))
            break
    if next_batch:
        break

if next_batch is None:
    print('EXTENSION COMPLETE: all 120 behavior-level runs have checkpoints.')
else:
    seed, start, count, remaining = next_batch
    print(f'Next batch: seed {seed}, behaviors {start+1}-{start+count}/20, '
          f'{remaining} behavior-level runs remaining in this batch', flush=True)
    subprocess.run([
        'python', '-u', 'run_experiment_resumable.py',
        '--mode', 'ollama', '--allow-local-tests', '--task', 'extension',
        '--seeds', str(seed), '--start', str(start), '--limit', str(count),
        '--data', 'data/harmful-behaviors.csv', '--out', str(EXT_FOLDER)
    ], check=True)

completed = len(list((EXT_FOLDER / 'checkpoints').glob('*.json')))
print(f'Extension progress: {completed}/120 behavior-level runs saved to Drive.')
print('Click this same cell again for the next batch.')


In [ ]:
import subprocess

subprocess.run([
    "python", "analyze_results.py",
    "--input", str(EXT_FOLDER)
], check=True)

In [ ]:

import pandas as pd

results = pd.read_csv(EXT_FOLDER / "results.csv")

summary = results.groupby(["setup", "seed"]).agg(
    behaviors=("success", "count"),
    success_rate=("success", "mean"),
    avg_queries=("queries_used", "mean")
)

summary["success_rate"] = (summary["success_rate"] * 100).round(2)
summary["avg_queries"] = summary["avg_queries"].round(2)

display(summary)


In [ ]:

successful = results[results["success"] == 1]

display(
    successful.groupby(["setup", "seed"])["queries_used"]
    .mean()
    .round(2)
    .rename("queries_per_success")
)


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(EXT_FOLDER / "results.csv")
fig_dir = EXT_FOLDER / "figures"
fig_dir.mkdir(exist_ok=True)

names = ["breadth", "depth"]
labels = ["Breadth (10x3)", "Depth (5x6)"]

# Figure 1: Success rate
rates = [
    100 * df[df["setup"] == name]["success"].mean()
    for name in names
]

plt.figure(figsize=(6, 4))
bars = plt.bar(labels, rates, color=["#5B8DEF", "#47A88A"], width=0.55)
plt.ylabel("Judge-assessed success rate (%)")
plt.ylim(0, 100)
plt.title("Breadth vs. Depth: Success Rate")

for bar, rate in zip(bars, rates):
    plt.text(bar.get_x() + bar.get_width()/2, rate + 2,
             f"{rate:.1f}%", ha="center")

plt.tight_layout()
plt.savefig(fig_dir / "success_rate.png", dpi=300)
plt.show()

# Figure 2: Queries per successful run
successful = df[df["success"] == 1]
qps = [
    successful[successful["setup"] == name]["queries_used"].mean()
    for name in names
]

plt.figure(figsize=(6, 4))
bars = plt.bar(labels, qps, color=["#5B8DEF", "#47A88A"], width=0.55)
plt.ylabel("Queries per success")
plt.title("Breadth vs. Depth: Query Efficiency")
plt.ylim(0, max(qps) * 1.25)

for bar, value in zip(bars, qps):
    plt.text(bar.get_x() + bar.get_width()/2, value + 0.3,
             f"{value:.2f}", ha="center")

plt.tight_layout()
plt.savefig(fig_dir / "query_efficiency.png", dpi=300)
plt.show()

# Figure 3: Success as query budget increases
plt.figure(figsize=(7, 4))

for name, label in zip(names, labels):
    subset = df[df["setup"] == name]
    rates = []

    for budget in range(1, 31):
        found = (
            (subset["success"] == 1) &
            (subset["queries_used"] <= budget)
        )
        rates.append(100 * found.mean())

    plt.plot(range(1, 31), rates,
             marker="o", markersize=3, label=label)

plt.xlabel("Target query budget")
plt.ylabel("Cumulative success rate (%)")
plt.title("Success Rate vs. Query Budget")
plt.xlim(1, 30)
plt.ylim(0, 100)
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(fig_dir / "success_vs_queries.png", dpi=300)
plt.show()

print("Figures saved to:", fig_dir)


### Extension progress and analysis

View progress whenever you want. To avoid drawing conclusions from incomplete data, the figures are generated **only after all 120 runs are saved**.

In [ ]:
import subprocess
from pathlib import Path
saved = len(list((EXT_FOLDER / 'checkpoints').glob('*.json')))
print(f'Extension: {saved}/120 checkpoints on Google Drive')
if saved == 120:
    subprocess.run(['python', 'analyze_results.py', '--input', str(EXT_FOLDER)], check=True)
else:
    print('Not finished yet — keep running the one-batch extension cell.')


## 9. Reproduction: run ONE resumable batch each time

**Same planned experiment:** 100 behaviors × 30 streams × 3 iterations × seed 0 = **100 behavior-level runs**.

Reproduction may be considerably slower, so this defaults to **2 behaviors per batch**. Repeated clicks advance automatically, and individual completed behaviors are saved to Drive. Restarting Colab doesn't reset saved progress.

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive")

print("PAIR folders:")
for folder in root.iterdir():
    if "PAIR" in folder.name:
        print(folder.name, "|", folder.is_dir())

REP_FOLDER = root / "PAIR_Project_Results/local_reproduction"
checkpoints = REP_FOLDER / "checkpoints"

print("Folder exists:", checkpoints.exists())
print("Saved behaviors:", len(list(checkpoints.glob("*.json"))))

In [ ]:
import requests

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print("Ollama status:", r.status_code)
    print("Models:", [m["name"] for m in r.json().get("models", [])])
except Exception as e:
    print("Ollama error:", e)

checkpoints = REP_FOLDER / "checkpoints"
print("Saved behaviors:", len(list(checkpoints.glob("*.json"))))
print("Folder:", REP_FOLDER)

In [ ]:
!tail -n 80 /tmp/ollama.log

In [ ]:
import subprocess
from run_experiment_resumable import load_behaviors, checkpoint_path

repro_batch_size = 1       # Change to 1 if your GPU session is especially short
all_behaviors = load_behaviors('data/harmful-behaviors.csv')
REP_FOLDER.mkdir(parents=True, exist_ok=True)
next_batch = None

for start in range(0, len(all_behaviors), repro_batch_size):
    chunk = all_behaviors[start:start + repro_batch_size]
    if any(not checkpoint_path(REP_FOLDER, 'reproduction', 0, b['id']).exists() for b in chunk):
        next_batch = (start, len(chunk))
        break

if next_batch is None:
    print('REPRODUCTION COMPLETE: all 100 behaviors have checkpoints.')
else:
    start, count = next_batch
    print(f'Next reproduction batch: behaviors {start+1}-{start+count}/100', flush=True)
    subprocess.run([
        'python', '-u', 'run_experiment_resumable.py',
        '--mode', 'ollama', '--allow-local-tests', '--task', 'reproduction',
        '--seeds', '0', '--start', str(start), '--limit', str(count),
        '--data', 'data/harmful-behaviors.csv', '--out', str(REP_FOLDER)
    ], check=True)

completed = len(list((REP_FOLDER / 'checkpoints').glob('*.json')))
print(f'Reproduction progress: {completed}/100 behaviors saved to Drive.')
print('Click this cell again for the next batch.')


### Reproduction progress and analysis

In [ ]:
import subprocess
saved = len(list((REP_FOLDER / 'checkpoints').glob('*.json')))
print(f'Reproduction: {saved}/100 checkpoints on Google Drive')
if saved == 100:
    subprocess.run(['python', 'analyze_results.py', '--input', str(REP_FOLDER)], check=True)
else:
    print('Not finished yet — keep running the reproduction batch cell.')


## 10. View figures and download a copy

The main results already live in Drive. Downloading a ZIP is optional. Figures appear once you finish and run analysis.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for folder in (EXT_FOLDER, REP_FOLDER):
    figures = sorted((folder / 'figures').glob('*.png'))
    if figures:
        print('Figures from', folder)
        for f in figures:
            display(Image(filename=str(f)))
    else:
        print('No figures yet for', folder, '(finish experiment and run analysis)')


In [ ]:
import shutil
from google.colab import files
# Change to REP_FOLDER to download reproduction instead.
folder = EXT_FOLDER
if folder.exists():
    zipname = shutil.make_archive('/content/PAIR_results_copy', 'zip', folder)
    files.download(zipname)
else:
    print('No results folder yet')


## Reporting checklist

- **Reproduction:** 100 behaviors; 30 streams × 3 iterations; one target query per candidate; seed 0.
- **Extension:** two fixed conditions, 10×3 vs 5×6, on the same 20 behaviors; seeds 0/1/2.
- **Models:** Ollama `llama3.2:3b` for attacker and target, and `llama-guard3:1b` as judge **unless you manually changed tags**. These are substitutes; do not claim an exact reproduction of the paper's Mixtral/Vicuna configuration.
- **Hardware:** report the Colab GPU shown in step 4 (e.g., Tesla T4 when assigned), not a GPU from a previous session.
- **Metrics:** check `results.csv`, `attempts.csv`, and `run_info.json` on Drive. Plots/summary tables are regenerated after the full run.
- **Privacy:** output stores hashes and labels by default; do not enable `--save-text` or share raw unsafe model responses publicly.
- **Method limitation:** round-robin stream scheduling is sequential, not concurrent; safety-judge labels alone may overcount successful task completions. Label results as **judge-assessed unsafe rate**, and manually review a sample before claiming a verified jailbreak rate.

**After reconnecting:** rerun setup (upload ZIP, mount Drive, create the runner, download dataset, start Ollama/pull missing models), then run the next batch cell. Google Drive checkpoints survive Colab runtime resets as long as they finish syncing.